In [4]:
import os
import shutil
import random
import csv
import numpy as np
import cv2 as cv
from skimage import io
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
import warnings

In [5]:
def gen_train_test(container_dir, max_category_sample=500):
    data_dir = 'data'
    train_dir = os.path.join(data_dir, 'train')
    test_dir = os.path.join(data_dir, 'test')
    labels_file = os.path.join(data_dir, 'test_labels.csv')

    if os.path.exists(data_dir):
        shutil.rmtree(data_dir)

    os.makedirs(train_dir)
    os.makedirs(test_dir)

    test_labels = []

    for class_name in os.listdir(container_dir):
        class_path = os.path.join(container_dir, class_name)
        if not os.path.isdir(class_path):
            continue
            
        train_class_dir = os.path.join(train_dir, class_name)
        os.makedirs(train_class_dir)

        files = os.listdir(class_path)
        random.shuffle(files)

        train_files = files[:max_category_sample]
        test_files = files[max_category_sample:]

        for file in train_files:
            src = os.path.join(class_path, file)
            dst = os.path.join(train_class_dir, file)
            shutil.copy(src, dst)

        for file in test_files:
            src = os.path.join(class_path, file)
            new_test_filename = f"{class_name}_{file}"
            dst = os.path.join(test_dir, new_test_filename)
            shutil.copy(src, dst)
            test_labels.append([new_test_filename, class_name])

    # Zapis do CSV
    with open(labels_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'label'])
        writer.writerows(test_labels)
        


In [6]:
def process_single_image(img_path, newSize, interpol, colorConv, stand, ignore_image_a, norm):
    img = io.imread(img_path)
    
    if ignore_image_a and img.shape[-1] == 4:
        img = img[:, :, :3]
        
    if colorConv is not None:
        img = cv.cvtColor(img, colorConv)
        
    img = cv.resize(img, newSize, interpolation=interpol)
    img = img.astype(np.float32)
    
    if stand:
        mean, std = np.mean(img), np.std(img)
        img = (img - mean) / (std + 1e-8)
    elif norm:
        img = img / 255.0
        
    return img.flatten()

def load_train_images(container_path, newSize=(64, 64), interpol=cv.INTER_AREA, 
                      colorConv=None, stand=False, ignore_image_a=True, norm=True, max_sample=200):
    data, labels, categories_name = [], [], []
    
    categories = [d for d in os.listdir(container_path) if os.path.isdir(os.path.join(container_path, d))]
    categories_name = sorted(categories)
    
    for class_name in categories_name:
        class_path = os.path.join(container_path, class_name)
        files = os.listdir(class_path)
        random.shuffle(files)
        
        selected_files = files[:max_sample]
        
        for file in selected_files:
            img_path = os.path.join(class_path, file)
            try:
                flat_img = process_single_image(img_path, newSize, interpol, colorConv, stand, ignore_image_a, norm)
                data.append(flat_img)
                labels.append(class_name)
            except Exception as e:
                print(f"Błąd wczytywania pliku {img_path}: {e}")

    X = {
        "data": np.array(data),
        "categories_name": categories_name,
        "categories_count": len(categories_name),
        "labels": labels
    }
    return X

In [7]:
def load_test_images(container_path, newSize=(64, 64), interpol=cv.INTER_AREA, 
                     colorConv=None, stand=False, ignore_image_a=True, norm=True, max_sample=500):
    data, labels = [], []
    labels_file = os.path.join(os.path.dirname(container_path), 'test_labels.csv')
    
    file_to_label = {}
    with open(labels_file, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            file_to_label[row[0]] = row[1]
            
    class_files = {}
    for filename, label in file_to_label.items():
        if label not in class_files:
            class_files[label] = []
        class_files[label].append(filename)
        
    categories_name = sorted(list(class_files.keys()))
    samples_per_class = max_sample // len(categories_name)
    
    for class_name in categories_name:
        files = class_files[class_name]
        random.shuffle(files)
        selected_files = files[:samples_per_class]
        
        for file in selected_files:
            img_path = os.path.join(container_path, file)
            if os.path.exists(img_path):
                try:
                    flat_img = process_single_image(img_path, newSize, interpol, colorConv, stand, ignore_image_a, norm)
                    data.append(flat_img)
                    labels.append(class_name)
                except Exception as e:
                    pass

    X = {
        "data": np.array(data),
        "categories_name": categories_name,
        "categories_count": len(categories_name),
        "labels": labels
    }
    return X

In [8]:
gen_train_test(container_dir=r'C:\Users\dabro\Downloads\archive\plantvillage dataset\color', max_category_sample=500)

IMG_SIZE = (128, 128)

train_data_dict = load_train_images(
    container_path='data/train', 
    newSize=IMG_SIZE, 
    norm=True, 
    max_sample=200
)

test_data_dict = load_test_images(
    container_path='data/test', 
    newSize=IMG_SIZE, 
    norm=True, 
    max_sample=500
)

X_train = train_data_dict['data']
y_train_text = train_data_dict['labels']

X_test = test_data_dict['data']
y_test_text = test_data_dict['labels']

le = LabelEncoder()
y_train = le.fit_transform(y_train_text)
y_test = le.transform(y_test_text)

print(f"\nKształt X_train: {X_train.shape}")
print(f"Kształt X_test: {X_test.shape}")


Kształt X_train: (7552, 49152)
Kształt X_test: (493, 49152)


In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"Model: {model_name}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    print("-" * 40 + "\n")


print("OVR (One-vs-Rest)")
clf_ovr = OneVsRestClassifier(LogisticRegression(solver='liblinear', max_iter=500))
clf_ovr.fit(X_train, y_train)
y_pred_ovr = clf_ovr.predict(X_test)
evaluate_model('OVR', y_test, y_pred_ovr)

print("Multinomial (Softmax)")
clf_multi = LogisticRegression(solver='lbfgs', max_iter=500) 
clf_multi.fit(X_train, y_train)
y_pred_multi = clf_multi.predict(X_test)
evaluate_model('Multinomial', y_test, y_pred_multi)

OVR (One-vs-Rest)
